In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub sentencepiece protobuf matplotlib seaborn numpy


In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0)
gpu_name

'Tesla T4'

In [ ]:
gpu_mem = torch.cuda.get_device_properties(0).total_memory/1e9

In [ ]:
gpu_mem

15.637086208

In [ ]:
from huggingface_hub import login, whoami

login()
print(whoami()["name"])

HARINISHAANTHAN


In [ ]:
prompt_structure = [
    {
        "role": "system",
        "content": "You are a helpful assistant who answers concisely."
    },
    {
        "role": "user",
        "content": "What's the capital of France?"
    },
    {
        "role": "assistant",
        "content": "Paris"
    }
]


# system
# user
# assistant

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_name = "google/gemma-3-4b-it"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from transformers import TextStreamer


In [ ]:
question = "what is the capital of India"


def chat(user_message, system_prompt):
  message = [
      {"role": "system", "content" : system_prompt},
      {"role": "user", "content" : user_message}
  ]

  input_text = tokenizer.apply_chat_template(message, add_generation_prompt = True, tokenize = False)
  inputs = tokenizer(input_text, return_tensors = "pt").to(model.device)

  streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)

  with torch.no_grad():
    model.generate(
        **inputs,
        max_new_tokens = 300,
        do_sample = True,
        temperature = 1.5,
        streamer=streamer,
        top_k = 50
    )

In [ ]:
chat("who is the cm of tamilnadu", "You are a helpful assistant") #1B


As of today, November 2, 2023, the Chief Minister of Tamil Nadu is **M. K. Stalin**. 

He assumed office on May 7, 2021.

You can always find the most up-to-date information on the Tamil Nadu government's official website: [https://tn.gov.in/](https://tn.gov.in/)


In [ ]:
question = "If you have 3 apples and take away 2, how many apples do you have?"
print(question)
print("Response:")
chat(question, "You are a helpful assistant")


If you have 3 apples and take away 2, how many apples do you have?
Response:
You would have 1 apple. 😊 

3 - 2 = 1

